GSE226600 : AML Leukemic Stem Cells

source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE226600

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

In [2]:
!curl -sL -o GSE226600_RAW.tar "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE226600&format=file" && tar xf GSE226600_RAW.tar && echo "Done: $(ls *.h5 | wc -l) h5 files, $(ls *.tsv.gz 2>/dev/null | wc -l) annotation files"

Done: 6 h5 files, 4 annotation files


In [12]:
!gunzip -f *.gz

In [14]:
import gzip
samples = {
    'GSM7080011_t821-1_filtered_feature_bc_matrix.h5': ('t821-1', 'mixed'),
    'GSM7080012_t821-2_filtered_feature_bc_matrix.h5': ('t821-2', 'mixed'),
    'GSM8033681_t821-3-LSC-filtered_feature_bc_matrix.h5': ('t821-3', 'LSC'),
    'GSM8033682_t821-3-blast-filtered_feature_bc_matrix.h5': ('t821-3', 'Blast'),
    'GSM8033683_t821-4-LSC-filtered_feature_bc_matrix.h5': ('t821-4', 'LSC'),
    'GSM8033684_t821-4-blast-filtered_feature_bc_matrix.h5': ('t821-4', 'Blast'),
}

anno_files = {
    't821-1': 'GSM7080011_t821-1_scRNAseq_barcode_cluster_assignments.tsv',
    't821-2': 'GSM7080012_t821-2_scRNAseq_barcode_cluster_assignments.tsv',
}

adatas = []
for h5file, (sample, stype) in samples.items():
    adata = sc.read_10x_h5(h5file)
    adata.var_names_make_unique()
    adata.obs['sample'] = sample
    adata.obs['sorted_type'] = stype
    adata.obs['patient'] = sample.split('-')[0]
    
    if sample in anno_files:
        if anno_files[sample].endswith('.gz'):
            anno = pd.read_csv(anno_files[sample], sep='\t', compression='gzip')
        else:
            anno = pd.read_csv(anno_files[sample], sep='\t')
        anno.index = anno['CellBarcode']
        common = adata.obs.index.intersection(anno.index)
        adata.obs.loc[common, 'CellType'] = anno.loc[common, 'CellType'].values
        adata.obs.loc[common, 'CellCyclePhase'] = anno.loc[common, 'CellCyclePhase'].values
        adata.obs.loc[common, 'ClusterID'] = anno.loc[common, 'ClusterID'].values
    else:
        adata.obs['CellType'] = stype
        adata.obs['CellCyclePhase'] = np.nan
        adata.obs['ClusterID'] = np.nan
    
    adatas.append(adata)
    print(f'{sample} ({stype}): {adata.shape[0]} cells, {adata.shape[1]} genes')

adata = sc.concat(adatas, join='inner', label='library_id')
adata.obs_names_make_unique()
print(f'\nTotal: {adata.shape[0]} cells × {adata.shape[1]} genes')

# all cells are malignant
adata.obs['ground_truth_malignant'] = 1
print(f'Ground truth: 100% malignant ({adata.n_obs} cells)')

/home1/prashantp/venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


t821-1 (mixed): 3302 cells, 19941 genes


/home1/prashantp/venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home1/prashantp/venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


t821-2 (mixed): 2898 cells, 20582 genes
t821-3 (LSC): 1844 cells, 20582 genes


/home1/prashantp/venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


...

/home1/prashantp/venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


t821-4 (LSC): 4104 cells, 20582 genes


/home1/prashantp/venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


t821-4 (Blast): 6017 cells, 20582 genes

Total: 19419 cells × 19831 genes
Ground truth: 100% malignant (19419 cells)


/home1/prashantp/venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [15]:
print('Cell type distribution:')
print(adata.obs['CellType'].value_counts())
print()
print('Sample distribution:')
print(adata.obs['sample'].value_counts())
print()
print('Patient distribution:')
print(adata.obs['patient'].value_counts())

Cell type distribution:
CellType
Blast        9994
LSC          7533
LSC/Blast     727
Name: count, dtype: int64

Sample distribution:
sample
t821-4    10121
t821-1     3302
t821-3     3098
t821-2     2898
Name: count, dtype: int64

Patient distribution:
patient
t821    19419
Name: count, dtype: int64


In [17]:
from aeacus import Profiler

aeacus_profiler = Profiler(
    test_input=adata.copy(),
    norm_type='cpm_log1p'
)
aeacus_profiler.load()
aeacus_adata = aeacus_profiler.profile()

print('aeacus results:')
print(aeacus_adata.obs[['malignancy_call', 'malignancy_score']].describe())

malignant_mask = aeacus_adata.obs["malignancy_call"] == "Malignant"
n_malignant = malignant_mask.sum()
n_total = len(aeacus_adata)
percent = 100 * malignant_mask.mean()

print(f'\nCalled malignant: {n_malignant} / {n_total} ({percent:.1f}%)')

Model features: 3778
Missing features: 43 (1.14%)
aeacus results:
       malignancy_score
count      19419.000000
mean           0.354403
std            0.264758
min            0.000885
25%            0.113770
50%            0.309588
75%            0.557080
max            0.984082

Called malignant: 5799 / 19419 (29.9%)


## 3. scMalignantFinder

In [20]:
import sys
import importlib.util

# Make local package imports (utils.py, etc.) work
sys.path.insert(0, "/home1/prashantp/scMalignantFinder/scMalignantFinder")

# Load classifier.py directly
spec = importlib.util.spec_from_file_location(
    "classifier",
    "/home1/prashantp/scMalignantFinder/scMalignantFinder/classifier.py",
)
classifier = importlib.util.module_from_spec(spec)
spec.loader.exec_module(classifier)

scmf = classifier.scMalignantFinder(
    test_input=adata.copy(),
    pretrain_dir="/home1/prashantp/scMalignantFinder/model",
    norm_type=True,
    n_thread=1,
    use_raw=False,
)

scmf.load()
scmf_adata = scmf.predict()

print("scMalignantFinder results:")
print(scmf_adata.obs["scMalignantFinder_prediction"].value_counts())

print("\nMalignancy probability:")
print(scmf_adata.obs["malignancy_probability"].describe())

/home1/prashantp/venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Model features: 2707
Missing features: 33 (1.22%)
scMalignantFinder results:
scMalignantFinder_prediction
Normal       16737
Malignant     2682
Name: count, dtype: int64

Malignancy probability:
count    1.941900e+04
mean     2.953676e-01
std      1.831821e-01
min      1.801693e-20
25%      1.671022e-01
50%      2.764934e-01
75%      4.080524e-01
max      9.941215e-01
Name: malignancy_probability, dtype: float64
